# Benchmark Generation
**Ranges:** Bucketed by Infix Closure Size (ICSize)

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd

CONFIGS = [
    {
        "range": (16, 127),
        "NUM_WORDS_BASE": 10,
        "NUM_WORDS_RANGE": 3,
        "MAX_LENGTH_TYPE1": 7,
        "MAX_LENGTH_TYPE2": 10,
        "TARGET_COUNT": 50
    },
    {
        "range": (128, 255),
        "NUM_WORDS_BASE": 11,
        "NUM_WORDS_RANGE": 3,
        "MAX_LENGTH_TYPE1": 10,
        "MAX_LENGTH_TYPE2": 13,
        "TARGET_COUNT": 50
    },
    {
        "range": (256, 511),
        "NUM_WORDS_BASE": 13,
        "NUM_WORDS_RANGE": 3,
        "MAX_LENGTH_TYPE1": 13,
        "MAX_LENGTH_TYPE2": 18,
        "TARGET_COUNT": 100
    },
    {
        "range": (512, 1023),
        "NUM_WORDS_BASE": 14,
        "NUM_WORDS_RANGE": 3,
        "MAX_LENGTH_TYPE1": 18,
        "MAX_LENGTH_TYPE2": 24,
        "TARGET_COUNT": 100
    },
    {
        "range": (1024, 2047),
        "NUM_WORDS_BASE": 15,
        "NUM_WORDS_RANGE": 3,
        "MAX_LENGTH_TYPE1": 24,
        "MAX_LENGTH_TYPE2": 29,
        "TARGET_COUNT": 200
    }
]

ALPHABETS = [
    ['a', 'b'],
    ['a', 'b', 'c'],
    ['a', 'b', 'c', 'd']
]

OUTPUT_DIR = "benchmarks/extended"
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [ ]:
def InfixesOf(word):
    ic = set()
    for i in range(len(word) + 1):
        for j in range(len(word) - i + 1):
            ic.add(word[j:i + j])
    return ic

def GetICSize(pos, neg):
    ic = set()
    for w in pos: ic |= InfixesOf(w)
    for w in neg: ic |= InfixesOf(w)
    return len(ic)

def RandomWordType1(alphabet, maxLen):
    a = len(alphabet)
    weights = []
    choices = []
    for l in range(maxLen, -1, -1):
        w = (1 / a) ** (maxLen - l)
        if w < 1e-8:
            break
        weights.append(w)
        choices.append(l)
    ln = random.choices(choices, weights=weights)[0]
    return ''.join(random.choice(alphabet) for _ in range(ln))

def RandomWordType2(alphabet, maxLen):
    return ''.join(random.choice(alphabet) for _ in range(random.randint(0, maxLen)))

def GeneratePosNegSets(posNum, negNum, alphabet, maxLen, mode="type1"):
    total = posNum + negNum
    if total == 0: return [], []
    generator = RandomWordType1 if mode == "type1" else RandomWordType2
    used = set()
    attempts = 0
    while len(used) < total and attempts < total * 100:
        used.add(generator(alphabet, maxLen))
        attempts += 1
    if len(used) < total: return [], []
    used = list(used)
    random.shuffle(used)
    return used[:posNum], used[posNum:]

def FormatExample(t, exampleNum, pos, neg):
    p_lines = "\n".join(f'"{p}"' for p in pos)
    n_lines = "\n".join(f'"{n}"' for n in neg)
    return f"Type {t}, Exp {exampleNum}\n++\n{p_lines}\n--\n{n_lines}\n"

def get_scaled_params(t, config):
    if t == 1:
        max_len = config["MAX_LENGTH_TYPE1"]
    else:
        max_len = config["MAX_LENGTH_TYPE2"]
    num_words = random.randint(config["NUM_WORDS_BASE"] - config["NUM_WORDS_RANGE"], config["NUM_WORDS_BASE"] + config["NUM_WORDS_RANGE"])
    return max_len, num_words

In [ ]:
def GenerateBenchmarks():
    start_time = time.time()
    stats = []
    
    print(f"Starting generation...\n")
    
    for alphabet in ALPHABETS:
        alphabet_str = "".join(alphabet)
        for t in [1, 2]:
            mode = "type1" if t == 1 else "type2"
            for config in CONFIGS:
                low, high = config["range"]
                target_count = config["TARGET_COUNT"]
                
                dir_path = os.path.join(OUTPUT_DIR, alphabet_str, f"{low}-{high}", f"type{t}")
                os.makedirs(dir_path, exist_ok=True)
                
                count = 0
                attempts = 0
                print(f"Generating: Alphabet={alphabet_str}, Type={t}, Range={low}-{high}")
                
                while count < target_count:
                    attempts += 1
                    if attempts > target_count * 500:
                        print(f"  Giving up after {attempts} attempts. Generated {count}.")
                        break
                        
                    max_len, num_words = get_scaled_params(t, config)
                    num_pos = random.randint(num_words - config["NUM_WORDS_RANGE"], num_words + config["NUM_WORDS_RANGE"])
                    num_neg = random.randint(num_words - config["NUM_WORDS_RANGE"], num_words + config["NUM_WORDS_RANGE"])
                    max_len = random.randint(4, max_len)
                    
                    pos, neg = GeneratePosNegSets(num_pos, num_neg, alphabet, max_len, mode=mode)
                    if not pos and not neg: continue
                    
                    ic_size = GetICSize(pos, neg)
                    if low <= ic_size <= high:
                        count += 1
                        filename = f"type{t}_exp{count}.txt"
                        filepath = os.path.join(dir_path, filename)
                        with open(filepath, "w") as f:
                            f.write(FormatExample(t, count, pos, neg))
                        
                        if count % 50 == 0:
                            print(f"  Progress: {count}/{target_count}")
                
                stats.append({
                    "Alphabet": alphabet_str,
                    "Type": t,
                    "Range": f"{low}-{high}",
                    "Count": count
                })
    
    elapsed = time.time() - start_time
    print(f"\nAll generation finished in {elapsed:.2f}s")
    return pd.DataFrame(stats)

df_summary = GenerateBenchmarks()
print("\nGeneration Summary:")
print(df_summary.pivot(index=['Alphabet', 'Range'], columns='Type', values='Count'))